In [1]:
import pandas as pd
import numpy as np
from rapidfuzz.fuzz import ratio
import re
import src_any_abogados_fuzzy_merge_df_er_v1_sec as src
import src_any_abogados_numbers_merge_df_er_v2_sec as src2
import src_any_abogados_email_merge_df_er_v1_sec as src3

In [2]:
pd.set_option("display.max_rows", None)

In [3]:
def remove_empty_columns(df):

    df_aux = df.copy()
    empty_columns = list(df_aux.columns[df_aux.isna().mean() == 1])   # Almacena en una lista las columnas que están totalmente vacias
    if "Unnamed: 0" in df_aux.columns:
        empty_columns.append("Unnamed: 0")
    df_aux = df_aux.drop(columns=empty_columns)
    return df_aux

In [4]:
#Lectura de tablas
df_gm_o = pd.read_csv("datasets/Dataset_GM_PR.csv")
df_ca_o = pd.read_csv("datasets/datos_abogados.csv") 
df_naics_o = pd.read_csv("datasets/NAICS_Puerto Rico.csv")

df_gm_o = remove_empty_columns(df_gm_o)
df_ca_o = remove_empty_columns(df_ca_o)
df_naics_o = remove_empty_columns(df_naics_o)

df_gm_o["Email"] = df_gm_o["Email"].str.strip().replace(["Not available"], np.nan)

# Agregar columna que indique de dónde viene el registro
df_gm_o["dataset"] = "gm"
df_ca_o["dataset"] = "ca"
df_naics_o["dataset"] = "naics"

In [5]:
#Agregar columna "is Firm" para diferenciar Firma de Abogados
df_gm_o['is Firm'] = df_gm_o['Name'].apply(lambda x: bool(re.search(src.pattern_firma, str(x), flags=re.IGNORECASE)))

In [6]:
#DataFrame que almacena los registros que son firmas
df_gm_firmas = df_gm_o[df_gm_o["is Firm"]]

#DataFrame filtrado y almacena los registro que son abogados
df_gm_o = df_gm_o[~df_gm_o["is Firm"]]

#Normalización de nombres
df_gm_o["Clean_Name"] = df_gm_o["Name"].apply(lambda x: src.name_normalized(x)).str.lower()
df_ca_o["FULL NAME"] = df_ca_o["FULL NAME"].apply(lambda x: src.name_normalized(x)).str.lower()
df_naics_o["Name_N"] = df_naics_o["Name_N"].apply(lambda x: src.name_normalized(x)).str.lower()

#Reset index
df_gm_o = df_gm_o.reset_index(drop=True)
df_ca_o = df_ca_o.reset_index(drop=True)
df_naics_o = df_naics_o.reset_index(drop=True)

In [7]:
# Columnas importantes: Name, city, state_name, Address, Website,Phone,Google_category,Type,Email,Specialization,Education,ExperiencE
# Columnas a Analizar: zip, state_id, Orig Specialization
# NO Columnas importantes: Google_URL, population,Google_rank,Google_opinions,Google_category,Processed,Name AI

df_gm_o.isna().mean()

Name                   0.000000
Google_URL             0.000000
zip                    0.000000
city                   0.000000
state_id               0.000000
state_name             0.000000
population             0.000000
Address                0.067335
Website                0.659026
Phone                  0.085960
Google_rank            0.193410
Google_opinions        0.177650
Google_category        0.021490
Name len               0.000000
Type                   0.000000
Email                  0.793696
Orig Specialization    0.226361
Specialization         0.226361
Education              0.226361
Experience             0.226361
Processed              0.226361
Name AI                0.226361
dataset                0.000000
is Firm                0.000000
Clean_Name             0.000000
dtype: float64

In [8]:
# Columnas importantes: FULL NAME', 'FNAME', 'LNAME
# Columnas a Analizar: 
# NO Columnas importantes:

df_ca_o.isna().mean()

FULL NAME          0.000000
FNAME              0.000000
LNAME              0.000000
colegiacion        0.092222
rua                0.092593
correo             0.884444
tel_residencial    0.994074
tel_oficina        0.914074
tel_celular        0.957037
otro               0.994444
especialidades     0.635185
Practice Area      0.635185
delegacion         0.001481
State              0.001481
dataset            0.000000
dtype: float64

In [9]:
# Columnas importantes: FULL NAME', 'FNAME', 'LNAME
# Columnas a Analizar: 
# NO Columnas importantes:

df_naics_o.isna().mean()

Name_N                                      0.000000
ZoomInfo Contact ID                         0.000000
Last Name                                   0.000000
First Name                                  0.000000
Middle Name                                 0.517479
Job Title                                   0.127648
Job Title Hierarchy Level                   0.319915
Management Level                            0.686970
Job Start Date                              0.018008
Job Function                                0.462394
Department                                  0.396716
Direct Phone Number                         0.645657
Email Address                               0.253178
Email Domain                                0.253708
Mobile phone                                0.612818
Highest Level of Education                  0.658369
Contact Accuracy Score                      0.000000
Contact Accuracy Grade                      0.000000
ZoomInfo Contact Profile URL                0.

In [10]:
df_ca_o = src2.combine_columns_by_priority(df_ca_o,["tel_celular","tel_residencial","tel_oficina","otro"],"telefono")
df_naics_o = src2.combine_columns_by_priority(df_naics_o,["Mobile phone","Direct Phone Number"], "phone")

In [11]:
df_ca_o["telefono"] = df_ca_o["telefono"].apply(lambda phone: src2.normalized_phone(str(phone)) if pd.notnull(phone) else None)
df_naics_o["phone"] = df_naics_o["Mobile phone"].apply(lambda phone: src2.normalized_phone(str(phone)) if pd.notnull(phone) else None)

# Merge por coincidencia exacta de número de contacto

No se obtuvo ninguna coincidencia exacta por número de teléfono entre los 3 datasets.
Se obtuvo 3 coincidencias exactas entre los datasets del colegio de abogados y naics

In [12]:
#Merge entre gm y ca
df_gm_y_df_ca = src2.merge_by_contact_number(df_gm_o, df_ca_o, "Phone", "telefono", "Clean_Name", "FULL NAME")
print(df_gm_y_df_ca[["Clean_Name","FULL NAME","Phone","telefono","score"]].head())

#Merge entre gm y naics
df_gm_y_df_naics = src2.merge_by_contact_number(df_gm_o, df_naics_o, "Phone", "phone", "Clean_Name", "Name_N")
print(df_gm_y_df_naics[["Clean_Name","Name_N","Phone","phone","score"]].head())

#Merge entre ca y naics  -->> **  # Salen 3 registros coincidencia exacta **
df_ca_y_df_naics = src2.merge_by_contact_number(df_ca_o, df_naics_o, "telefono", "phone", "FULL NAME", "Name_N")
print(df_ca_y_df_naics[["FULL NAME","Name_N","telefono","phone","score"]].head())               

#Merge COMPLETO
df_merged_full = src2.merge_by_contact_number(df_gm_y_df_ca, df_naics_o, "Phone", "phone", "FULL NAME", "Name_N")
print(df_merged_full[["Clean_Name","FULL NAME","Phone","telefono","score"]].head())


Empty DataFrame
Columns: [Clean_Name, FULL NAME, Phone, telefono, score]
Index: []
Empty DataFrame
Columns: [Clean_Name, Name_N, Phone, phone, score]
Index: []
                 FULL NAME                           Name_N    telefono  \
0     esteban mujica cotto              esteban mujicacotto  7875184101   
1  josue castellanos otero  josue emanuel castellanos otero  7872995935   
2  francisco garcia garcia                 francisco garcia  7873984898   

        phone     score  
0  7875184101  0.974359  
1  7872995935  0.851852  
2  7873984898  0.820513  
Empty DataFrame
Columns: [Clean_Name, FULL NAME, Phone, telefono, score]
Index: []


# Merge por coincidencia exacta de Email

In [13]:
#Merge entre gm y ca    -->> ** 5 coincidencias exactas **
df_gm_y_df_ca = src3.merge_by_email(df_gm_o, df_ca_o, "Email", "correo", "Clean_Name", "FULL NAME")
print(df_gm_y_df_ca[["Clean_Name","FULL NAME","Email","correo","score"]].head())

#Merge entre gm y naics -->> NO COINCIDENCIAS
df_gm_y_df_naics = src3.merge_by_email(df_gm_o, df_naics_o, "Email", "Email Address", "Clean_Name", "Name_N")
print(df_gm_y_df_naics[["Clean_Name","Name_N","Phone","phone","score"]].head())

#Merge entre ca y naics  -->> ** 5 coincidencias exactas
df_ca_y_df_naics = src3.merge_by_email(df_ca_o, df_naics_o, "correo", "Email Address", "FULL NAME", "Name_N")
print(df_ca_y_df_naics[["FULL NAME","Name_N","correo","Email Address","score"]].head())               

#Merge COMPLETO   -->> NO COINCIDENCIAS
df_merged_full = src3.merge_by_email(df_gm_y_df_ca, df_naics_o, "correo", "Email Address", "FULL NAME", "Name_N")
print(df_merged_full[["Clean_Name","FULL NAME","Phone","telefono","score"]].head())

               Clean_Name               FULL NAME  \
0  lutgardo acevedo lopez  lutgardo acevedo lopez   
1        edil quiles seda        edil quiles seda   
2       omar bonet tirado       omar bonet tirado   
3       onix cintron baez       onix cintron baez   
4     beatriz cay vazquez     beatriz cay vazquez   

                         Email                       correo  score  
0        lacevedolaw@gmail.com        lacevedolaw@gmail.com    1.0  
1       quilesseda@hotmail.com       quilesseda@hotmail.com    1.0  
2          omar.bonet@capr.org          omar.bonet@capr.org    1.0  
3   onix.cintron.law@gmail.com   onix.cintron.law@gmail.com    1.0  
4  beatrizcayvazquez@gmail.com  beatrizcayvazquez@gmail.com    1.0  
Empty DataFrame
Columns: [Clean_Name, Name_N, Phone, phone, score]
Index: []
               FULL NAME                 Name_N                    correo  \
0   jose gonzalez rivera          jose gonzalez   jrg@gonzalezmorales.com   
1  ricardo garcia negron  ricardo ga

# Coincidencias exactas por Nombre

In [14]:
#Merge entre gm y ca    -->> ** 5 coincidencias exactas **
df_gm_y_df_ca = src3.merge_by_email(df_gm_o, df_ca_o, "Clean_Name", "FULL NAME", "Clean_Name", "FULL NAME")
print(df_gm_y_df_ca[["Clean_Name","FULL NAME","score"]].head())

#Merge entre gm y naics -->> ** 5 coincidencias exactas **
df_gm_y_df_naics = src3.merge_by_email(df_gm_o, df_naics_o, "Clean_Name", "Name_N", "Clean_Name", "Name_N")
print(df_gm_y_df_naics[["Clean_Name","Name_N","score"]].head())

#Merge entre ca y naics  -->> ** 5 coincidencias exactas **
df_ca_y_df_naics = src3.merge_by_email(df_ca_o, df_naics_o, "FULL NAME", "Name_N", "FULL NAME", "Name_N")
print(df_ca_y_df_naics[["FULL NAME","Name_N","score"]].head())               

#Merge COMPLETO   -->> NO HAY CONCIDENCIAS  
df_merged_full = src3.merge_by_email(df_gm_y_df_ca, df_naics_o, "Clean_Name", "Name_N", "FULL NAME", "Name_N")
print(df_merged_full[["Clean_Name","FULL NAME","score"]].head())

               Clean_Name               FULL NAME  score
0    luis mercado hidalgo    luis mercado hidalgo    1.0
1  lutgardo acevedo lopez  lutgardo acevedo lopez    1.0
2   jorge hernandez lopez   jorge hernandez lopez    1.0
3   angela oquendo negron   angela oquendo negron    1.0
4   manlio arraiza donate   manlio arraiza donate    1.0
         Clean_Name            Name_N  score
0  felipe sotoortiz  felipe sotoortiz    1.0
1     jaime ruberte     jaime ruberte    1.0
2   gilberto oliver   gilberto oliver    1.0
3      jose benitez      jose benitez    1.0
4   francisco ramos   francisco ramos    1.0
                      FULL NAME                        Name_N  score
0   veronica gonzalez rodriguez   veronica gonzalez rodriguez    1.0
1          edwin rivera cintron          edwin rivera cintron    1.0
2             alba lopez arzola             alba lopez arzola    1.0
3       isabel ruberte figueroa       isabel ruberte figueroa    1.0
4  isamillie melendez caraballo  isamillie 

# Merge por coincidencias difusas de nombre

In [15]:
df_gm = df_gm_o[["Clean_Name"]]
df_ca = df_ca_o[["FULL NAME"]]
df_naics = df_naics_o[["Name_N"]]

In [16]:
src.find_fuzzy_matches(df_gm, df_ca,"Clean_Name", "FULL NAME", 0.8)

[('maria barrera rosario', 'lizandra torres rosario'),
 ('orlando cabrera rodriguez', 'carlos rivera rodriguez'),
 ('martha berrios rivera', 'omar colon rivera'),
 ('yolanda castro borrero', 'yolanda torres roque'),
 ('catherine catala lasanta', 'catherine ortiz martinez'),
 ('luis del valle colon', 'luis valladares montalvo'),
 ('juan encarnacion cruz', 'juan ojeda arnau'),
 ('ramonita figueroa collazo', 'jonathan florian collazo'),
 ('santiago gutierrez armstrong', 'torres santiago nevarez tsn'),
 ('jose melendez cox', 'luis melendez munoz'),
 ('julia santiago deliz', 'talina santiago rodriguez'),
 ('zamarie vazquez prieto', 'neomar vazquez torres'),
 ('miguel manzano rivera', 'miguel sanabria'),
 ('ruben lucena quiles', 'marie lou de la luz quiles'),
 ('norana sanchez alvarado', 'norana sanchez alvarado'),
 ('pedro berrios lara', 'veronica berrios martinez'),
 ('donald milan guindin', 'dorado legal studio'),
 ('ivelisse iguina de la rosa', 'luis quintana burgos'),
 ('carlos vera mun

In [17]:
ratio('maria barrera rosario', 'lizandra torres rosario')

68.18181818181819